# KAT-VAD v2 — **Nexar N2–N5**: extract → build → train `whole` → read val + DoTA-CAP → window trigger

Plan `.project/plans/katvad-v2-nexar-feasibility.md`, rules `core/docs/v2/PREREG_ADDENDUM.md` **§21** (D-N1…D-N10).
**`whole` first (D-N10):** `window` is trained only if the pilot's window trigger (W1–W3, step 7) fires.
Needs N0 on Drive (`Thesis/data/Nexar/raw/`) and the frozen split in the uploaded code (`core/splits/v2/nexar_*`).

| step | what | GPU? |
|---|---|---|
| 0 | setup, deps | — |
| 1 | preflight: split `--check`, raw mp4 present, §21 present | no |
| 2 | **N2** `nexar_extract`: every mp4 decoded once → CLIP `s1` + V2-S `s1` (all 1,500, label-free), resumable by chunk | **yes** |
| 3 | **N4** `nexar_build corpus` (`Nexar_whole`, `Nexar_window`) + `Nexar_s8_ncc` (A0) + `fit-ids` A3 bake + KNN per corpus | no |
| 4 | **N5** train `CONSTRUCTIONS` × {A0, A3} × `SEEDS`, step-matched (`NEXAR_STEP_BUDGET`), chunked, J10 | **yes** |
| 5 | read **`nexar_val`**: crops (endpoint) + whole videos, per construction × arm; **+ E3's T2-trained A3 zero-shot** (W2's bar) | yes |
| 6 | **second look** on DoTA-CAP-dev (opened set — descriptive only, D-N8) | yes |
| 7 | summary + **window trigger** (pilot only) → **paste `summary_<tag>.md` to Claude** | no |

**Run order:** pilot `SEEDS = [PILOT_SEED]` with `CONSTRUCTIONS = ['whole']` (2 runs) → paste the summary. Then:
* trigger **passes** → `SEEDS = RUN_SEEDS` (10 runs, `whole` only);
* trigger **fires** → add `'window'` to `CONSTRUCTIONS`, pilot it, then `RUN_SEEDS` for both (D-N9 applies).

Every step skips what is already verified on Drive. **`nexar_test` is never opened here.**

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read: data/ (Nexar raw, DoTA), cache/ (DoTA s1, DoTA-CAP V2-S)
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # write: kat-vad/ (code), data/, cache/, outputs/
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'NEXAR_STEP_BUDGET'), 'this upload predates Nexar N2-N5 -- re-upload branch v2'

DATASET    = constants.NEXAR_DATASET                       # 'Nexar' -> the T2 / DoTA definition set
KERNEL     = constants.DADA_ORIGIN_SCORE_HEAD_KERNEL       # 3 (E3 recipe)
TOPK_PCT   = constants.DADA_ORIGIN_MIL_TOPK_PCT            # 5
CRN        = 'R2'
ENC        = constants.VIDEOMAE_ENCODER_S                  # V2-S
MOTION_DIM = constants.VIDEOMAE_ARCH[ENC][0]               # 384
PILOT_SEED = constants.NEXAR_PILOT_SEED                    # 2099
RUN_SEEDS  = list(constants.NEXAR_RUN_SEEDS)               # 2024-2028
ARMS       = ['A0', 'A3']

# ---- this session --------------------------------------------------------------------------------
CONSTRUCTIONS = [constants.NEXAR_CONSTRUCTION_WHOLE]   # D-N10: add 'window' ONLY if the pilot trigger fired
SEEDS = [PILOT_SEED]          # pilot first; then RUN_SEEDS (D-N7)
TAG   = 'pilot' if SEEDS == [PILOT_SEED] else 'full'
CHUNK_EPOCHS = 10             # epochs per train subprocess (resume between chunks)
# --------------------------------------------------------------------------------------------------

RAW       = constants.DATA_ROOT / constants.NEXAR_DATASET / 'raw'
CENSUS    = constants.OUTPUT_ROOT / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'   # written by N0
V2_CACHE  = Path(DRIVE_V2) / 'cache'
CLIP_S1   = V2_CACHE / 'clip' / constants.NEXAR_CLIP_S1_DATASET
VIDEO_S1  = V2_CACHE / 'video' / ENC / f'{constants.NEXAR_VIDEO_S1_DATASET}_squash'
CLIP_S8   = V2_CACHE / 'clip' / 'Nexar_s8_ncc'
A3_BAKE   = V2_CACHE / 'v2' / f'A3_{CRN}_S' / 'Nexar_s8'
A3_CAP    = V2_CACHE / 'v2' / f'A3_{CRN}_S' / 'DoTA_CAP_s3_from_s1_nexar'   # Nexar stats on DoTA-CAP-dev
DATA      = {c: Path(DRIVE_V2) / 'data' / f'{DATASET}_{c}' for c in CONSTRUCTIONS}
KNN       = {c: V2_CACHE / 'knn' / f'{DATASET}_{c}' / constants.KNN_CACHE_FILENAME for c in CONSTRUCTIONS}
REPORTS   = constants.OUTPUT_ROOT / 'v2' / 'REPORTS'
RUNS      = constants.OUTPUT_ROOT / 'v2_nexar'                # <construction>/<arm>/s<seed>/
DOTA      = constants.DATA_ROOT / 'DoTA'
DOTA_S1   = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
CAP_S1    = constants.VIDEO_CACHE_DIR / ENC / f'{constants.DOTA_CAP_DATASET}_s1_squash'
E3_RUNS   = constants.OUTPUT_ROOT / 'v2_e3' / 'A3'                       # E3's T2-trained A3, s2024-2028 (read only)
E3_STATS  = V2_CACHE / 'v2' / f'A3_{CRN}_S' / constants.DADA_ORIGIN_DATASET   # their T2 statistics (fit dir)
E3_STEP   = constants.NEXAR_STEP_BUDGET                                   # = E3's 1,740 (J10)
LOCAL     = Path('/content/nexar')
LOCAL.mkdir(parents=True, exist_ok=True)
for name in ('RAW', 'CENSUS', 'CLIP_S1', 'VIDEO_S1', 'RUNS'):
    print(f'{name:9s}', globals()[name])
print('session seeds', SEEDS, '| tag', TAG)

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu
python -c "import transformers, av, einops, faiss; print('transformers', transformers.__version__)"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 2 and 4 will be unusably slow'
df -h /content | tail -1

In [ ]:
import json
import shutil
import subprocess
import time

import numpy as np
import torch

ENV = dict(os.environ)


def run(cmd, log=None, quiet_steps=False):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                if not (quiet_steps and 'global_step' in line):
                    print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    '''Copy a Drive dir (or only ``ids`` + its manifests) / file to VM disk once (bit-identical; C2).'''
    if dst.exists() and ids is None:
        return dst
    t0 = time.time()
    if src.is_dir() and ids is None:
        shutil.copytree(src, dst)
    elif src.is_dir():
        dst.mkdir(parents=True, exist_ok=True)
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for m in src.glob('*.json'):
            shutil.copy2(m, dst / m.name)
        for m in src.glob('*.npz'):
            shutil.copy2(m, dst / m.name)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    '''Copy a VM-local result to Drive and fsync every file (C10).'''
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        files = [p for p in dst.rglob('*') if p.is_file()]
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        files = [dst]
    for p in files:
        with open(p, 'rb+') as fh:
            os.fsync(fh.fileno())


def ckpt_step(path):
    if not path.exists():
        return 0
    return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path

## 1. Preflight

The split must be the committed one (`--check` recomputes it from the census); §21 must be in the uploaded addendum.

In [ ]:
from core.data.v2_splits import load_split

addendum = (REPO / 'core/docs/v2/PREREG_ADDENDUM.md').read_text(encoding='utf-8')
assert '## 21.' in addendum, 'upload predates addendum §21 (Nexar) -- re-upload branch v2'
assert CENSUS.is_file(), f'missing {CENSUS} -- run nexar_n0_census.ipynb first'
run([sys.executable, '-m', 'core.tools.nexar_splits', '--census', CENSUS, '--check'])
census = json.loads(CENSUS.read_text())
ALL_IDS = sorted(census['videos'])
TRAIN = load_split(constants.V2_SPLIT_NEXAR_TRAIN)
VAL = load_split(constants.V2_SPLIT_NEXAR_VAL)
missing = [v for v in ALL_IDS if not (RAW / census['videos'][v]['path']).is_file()]
assert not missing, f'{len(missing)} mp4 missing under {RAW}: {missing[:5]} -- re-run nexar_n0_census step 1'
print(f'{len(ALL_IDS)} videos on Drive; train {len(TRAIN)}, val {len(VAL)} (nexar_test sealed, not loaded)')

## 2. N2 — extract CLIP `s1` + V2-S `s1` (GPU, resumable)

All 1,500 videos: extraction reads no label, and the final read (§21) needs `nexar_test`'s features later. One decode
per mp4 feeds both encoders (same resize, two normalizations — pinned bit-exact by `test_nexar_pipeline.py`).
Chunks of `CHUNK_VIDEOS`; a chunk whose videos are all on Drive is skipped. **Re-run this cell after a disconnect.**
Expect ≈ 1.7 M frames: a few hours on an A100, longer on a T4.

In [ ]:
from core.tools.feature_cache import is_complete

CHUNK_VIDEOS = 100
N2 = REPORTS / 'nexar_n2'


def done(v):
    return is_complete(CLIP_S1 / f'{v}.npy') and is_complete(VIDEO_S1 / f'{v}.npy')


chunks = [ALL_IDS[i:i + CHUNK_VIDEOS] for i in range(0, len(ALL_IDS), CHUNK_VIDEOS)]
t_start = time.time()
for k, chunk in enumerate(chunks):
    todo = [v for v in chunk if not done(v)]
    if not todo:
        continue
    ids_file = write_ids(LOCAL / 'ids' / f'extract_{k:02d}.txt', todo)
    print(f'=== chunk {k + 1}/{len(chunks)}: {len(todo)} videos')
    run([sys.executable, '-m', 'core.tools.nexar_extract', '--root', RAW, '--census', CENSUS,
         '--ids-file', ids_file, '--encoder', ENC, '--clip-out', CLIP_S1, '--video-out', VIDEO_S1,
         '--report', N2 / f'extract_{k:02d}_{int(time.time())}.json'], log=LOCAL / 'extract.log')
left = [v for v in ALL_IDS if not done(v)]
print(f'extraction: {len(ALL_IDS) - len(left)} / {len(ALL_IDS)} done ({(time.time() - t_start) / 60:.0f} min this session)')
assert not left, f'{len(left)} videos still to extract: re-run this cell'

In [ ]:
# Gate: CLIP rows == V2-S rows for every video; decoded frames vs the census header (printed, C10)
rows_bad, header_bad = [], []
for v in ALL_IDS:
    n_x = np.load(CLIP_S1 / f'{v}.npy', mmap_mode='r').shape[0]
    n_u = np.load(VIDEO_S1 / f'{v}.npy', mmap_mode='r').shape[0]
    if n_x != n_u:
        rows_bad.append(v)
    if n_x != census['videos'][v]['frames_header']:
        header_bad.append((v, n_x, census['videos'][v]['frames_header']))
check = {'videos': len(ALL_IDS), 'clip_vs_motion_rows_mismatch': rows_bad, 'decoded_vs_header_mismatch': header_bad,
         'clip_manifest': json.loads((CLIP_S1 / 'clip_manifest.json').read_text()),
         'video_manifest': json.loads((VIDEO_S1 / constants.VIDEO_MANIFEST_FILENAME).read_text())}
N2.mkdir(parents=True, exist_ok=True)
(N2 / 'extract_check.json').write_text(json.dumps(check, indent=2))
print(f'rows mismatch {len(rows_bad)}, decoded-vs-header mismatch {len(header_bad)}: {header_bad[:5]}')
assert not rows_bad, 'CLIP and motion caches disagree (C13) -- paste extract_check.json to Claude'

## 3. N4 — corpora, A0 cache, A3 bake, KNN (CPU)

* `Nexar_whole` = every `nexar_train` video (both pools) · `Nexar_window` = T2 windows from positive videos only
  (both are built; only `CONSTRUCTIONS` are trained).
* `Nexar_s8_ncc` = `Nexar_s1_ncc[::8]` (A0 input and KNN key) for train + val.
* A3: `build_v2_inputs fit-ids` — CRN R2 + V2-S statistics fitted on `nexar_train`, train + val baked at `s1[::8]`.
  One bake serves both constructions (the arms differ only in what they train on).

In [ ]:
N4 = REPORTS / 'nexar_n4'
BUILD = N4 / 'build_report.json'
if not all((DATA[c] / constants.LABELS_TRAIN_FILENAME).exists() for c in CONSTRUCTIONS) or not BUILD.exists():
    # builds BOTH corpora (CPU, seconds): `window` is ready if the trigger fires
    run([sys.executable, '-m', 'core.tools.nexar_build', 'corpus', '--census', CENSUS, '--clip-s1-dir', CLIP_S1,
         '--out-root', Path(DRIVE_V2) / 'data', '--report', BUILD])
build = json.loads(BUILD.read_text())
TRAIN_VAL = write_ids(LOCAL / 'ids' / 'train_val.txt', sorted(TRAIN + VAL))
run([sys.executable, '-m', 'core.tools.nexar_build', 'subsample', '--clip-s1-dir', CLIP_S1,
     '--ids-file', TRAIN_VAL, '--stride', constants.NEXAR_TRAIN_STRIDE, '--out-dir', CLIP_S8])
if not (A3_BAKE / constants.V2_INPUT_MANIFEST_FILENAME).exists():
    run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'fit-ids',
         '--train-ids-file', REPO / 'core/splits/v2/nexar_train.txt', '--ids-file', TRAIN_VAL,
         '--clip-dir', CLIP_S1, '--video-dir', VIDEO_S1, '--stride', constants.NEXAR_TRAIN_STRIDE,
         '--crn', CRN, '--motion', ENC, '--out-dir', A3_BAKE])
for c in CONSTRUCTIONS:
    if not KNN[c].exists():
        KNN[c].parent.mkdir(parents=True, exist_ok=True)
        run([sys.executable, '-m', 'core.data.knn_cache', '--data-dir', DATA[c], '--dataset', DATASET,
             '--clip-dir', CLIP_S8, '--output', KNN[c]])
a3 = json.loads((A3_BAKE / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
print(f"A3 bake: s={a3['s']:.4f} c={a3['c']:.4f}, fitted on {a3['train_sources']} sources ({a3['train_ids_sha1'][:12]})")
print('val middle-ruler macro (position alone, whole videos):', build['val_middle_ruler_macro'])
for c in CONSTRUCTIONS:
    print(c, build['constructions'][c], '\n   plan', build['plan'][c])

## 4. N5 — train `CONSTRUCTIONS` × {A0, A3} × `SEEDS` (GPU, chunked, resumable)

E3 recipe (kernel 3, top-k 5 %, KIP off, stage 2, AMP) with **the same optimizer-step budget** for both constructions
(`epochs` from `build_report.json`). `whole` also sets `dvs.syn_max_num_clips = 3` (D-N6: a 5-clip splice of 40 s videos
overflows the 512-row cap and can truncate the anchor). Skip/resume reads the checkpoint's own `global_step` (J10).
Every run writes `run_manifest.json` (C17). **Re-run this cell after a disconnect.**

In [ ]:
MOTION = ['--set', f'v2.crn={CRN}', '--set', f'v2.motion={ENC}', '--set', f'model.motion_dim={MOTION_DIM}']
EXTRA = {'A0': [], 'A3': MOTION}
SYN = {'whole': ['--set', f'dvs.syn_max_num_clips={constants.NEXAR_WHOLE_SYN_MAX_CLIPS}'], 'window': []}
EPOCHS = {c: build['plan'][c]['epochs'] for c in CONSTRUCTIONS}

L_DATA = {c: stage(DATA[c], LOCAL / 'stage' / f'data_{c}') for c in CONSTRUCTIONS}
L_KNN = {c: stage(KNN[c], LOCAL / 'stage' / f'knn_{c}.npz') for c in CONSTRUCTIONS}
L_IN = {'A0': stage(CLIP_S8, LOCAL / 'stage' / 'clip_s8'), 'A3': stage(A3_BAKE, LOCAL / 'stage' / 'A3_s8')}


def run_dir(c, arm, seed):
    return RUNS / c / arm / f's{seed}'


def train_cmd(c, arm, seed, out_dir):
    return [sys.executable, '-m', 'core.train',
            '--set', 'train.stage=2', '--set', f'train.seed={seed}',
            '--set', f'train.num_epochs={EPOCHS[c]}', '--set', 'train.amp=true',
            '--set', f'data.dataset={DATASET}',
            '--set', f'model.score_head_kernel={KERNEL}', '--set', f'loss.mil_topk_pct={TOPK_PCT}',
            '--set', 'kip.enabled=false', *EXTRA[arm], *SYN[c],
            '--data-dir', L_DATA[c], '--clip-dir', L_IN[arm], '--knn-cache', L_KNN[c],
            '--output-dir', out_dir]


def verified(c, d):
    last = metrics_last(d / 'metrics.jsonl')
    return bool(last and last['epoch'] + 1 == EPOCHS[c]
                and ckpt_step(d / 'checkpoint_last.pt') == last['global_step'])


def sync(local, drive_dir):
    drive_dir.mkdir(parents=True, exist_ok=True)
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml', 'run_manifest.json'):
        if (local / name).exists():
            publish(local / name, drive_dir / name)


durations = []
todo = [(c, a, s) for s in SEEDS for c in CONSTRUCTIONS for a in ARMS if not verified(c, run_dir(c, a, s))]
print(f'{len(todo)} runs to train in this session; epochs {EPOCHS} (budget {constants.NEXAR_STEP_BUDGET} steps)')
for k, (c, arm, seed) in enumerate(todo):
    drive_dir, local = run_dir(c, arm, seed), LOCAL / 'runs' / c / arm / f's{seed}'
    local.mkdir(parents=True, exist_ok=True)
    last = metrics_last(drive_dir / 'metrics.jsonl')
    if not (local / 'checkpoint_last.pt').exists() and last and \
            ckpt_step(drive_dir / 'checkpoint_last.pt') == last['global_step']:
        for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
            shutil.copy2(drive_dir / name, local / name)          # resume from a consistent Drive copy
    (local / 'run_manifest.json').write_text(json.dumps({
        'construction': c, 'arm': arm, 'seed': seed, 'epochs': EPOCHS[c], 'plan': build['plan'][c],
        'data_dir': str(DATA[c]), 'clip_dir': str(CLIP_S8 if arm == 'A0' else A3_BAKE), 'knn': str(KNN[c]),
        'split_sha1': build['splits'], 'a3_train_ids_sha1': a3['train_ids_sha1'],
        'cmd': [str(x) for x in train_cmd(c, arm, seed, local)]}, indent=2))
    t0 = time.time()
    while True:
        done_ep = (metrics_last(local / 'metrics.jsonl') or {'epoch': -1})['epoch'] + 1
        if done_ep >= EPOCHS[c]:
            break
        cmd = train_cmd(c, arm, seed, local)
        if done_ep:
            cmd += ['--resume', local / 'checkpoint_last.pt']
        cmd += ['--stop-after-epochs', min(CHUNK_EPOCHS, EPOCHS[c] - done_ep)]
        print(f'=== {c}/{arm} s{seed}: epochs {done_ep} -> {min(done_ep + CHUNK_EPOCHS, EPOCHS[c])} of {EPOCHS[c]}')
        run(cmd, log=LOCAL / f'{c}_{arm}_s{seed}_train.log', quiet_steps=True)
        sync(local, drive_dir)
    sync(local, drive_dir)
    assert verified(c, drive_dir), f'{c}/{arm} s{seed}: J10 FAILS on Drive'
    durations.append(time.time() - t0)
    eta = sum(durations) / len(durations) * (len(todo) - k - 1)
    print(f'{c}/{arm} s{seed}: step {metrics_last(drive_dir / "metrics.jsonl")["global_step"]}, '
          f'{durations[-1] / 60:.1f} min; ETA this session {eta / 3600:.1f} h')
missing = [(c, a, s) for s in SEEDS for c in CONSTRUCTIONS for a in ARMS if not verified(c, run_dir(c, a, s))]
print('all runs of this session verified' if not missing else f'still to train: {missing}')

## 5. Read `nexar_val` — crops (endpoint) + whole videos

One call per construction × arm with every seed of `SEEDS` (seed-averaged per frame). Crops: an 8 s window with the
event centre at 0.1 … 0.9, CRN re-referenced per crop; the middle ruler beside each placement shows what position alone
scores there. `nexar_test` stays sealed.

In [ ]:
L_VAL_X = stage(CLIP_S1, LOCAL / 'stage' / 'clip_s1_val', VAL)
L_VAL_U = stage(VIDEO_S1, LOCAL / 'stage' / 'video_s1_val', VAL)
L_A3_STATS = L_IN['A3']                    # its manifest + v2_input_stats.npz
VAL_OUT = {(c, a): REPORTS / 'nexar_val' / f'{TAG}_{c}_{a}' for c in CONSTRUCTIONS for a in ARMS}
for (c, arm), out in VAL_OUT.items():
    ready = [s for s in SEEDS if verified(c, run_dir(c, arm, s))]
    if len(ready) != len(SEEDS):
        print(f'{c}/{arm}: {len(ready)}/{len(SEEDS)} seeds trained -- skipped')
        continue
    if (out / 'nexar_readout.json').exists():
        print(f'{c}/{arm}: reusing')
        continue
    runs = [x for s in SEEDS for x in ('--run', f's{s}', run_dir(c, arm, s) / 'checkpoint_last.pt')]
    stats = ['--stats-dir', L_A3_STATS, '--video-s1-dir', L_VAL_U] if arm == 'A3' else []
    run([sys.executable, '-m', 'core.tools.nexar_eval', *runs, '--census', CENSUS, '--clip-s1-dir', L_VAL_X,
         *stats, '--data-dir', L_DATA[c], '--split', constants.V2_SPLIT_NEXAR_VAL, '--out-dir', out],
        log=LOCAL / f'{c}_{arm}_val.log')

### 5b. Zero-shot: E3's T2-trained A3 on `nexar_val` (W2's bar)

The five E3 A3 checkpoints (T2-trained, never saw Nexar), each Nexar item baked with **their own T2 statistics**
(not Nexar's). No training; run once, reused by every tag.

In [ ]:
ZS_OUT = REPORTS / 'nexar_val' / 'zeroshot_e3_A3'
E3_SEEDS = list(constants.V2_E3_SEEDS)
for s in E3_SEEDS:                                             # J10 on every E3 checkpoint
    d = E3_RUNS / f's{s}'
    step, last = ckpt_step(d / 'checkpoint_last.pt'), metrics_last(d / 'metrics.jsonl')
    assert last is not None and step == last['global_step'] == E3_STEP, f'E3 A3 s{s}: J10 (ckpt {step})'
if (ZS_OUT / 'nexar_readout.json').exists():
    print('zero-shot: reusing')
else:
    L_E3_STATS = stage(E3_STATS, LOCAL / 'stage' / 'e3_A3_stats', ids=[])    # manifest + stats only
    runs = [x for s in E3_SEEDS for x in ('--run', f's{s}', E3_RUNS / f's{s}' / 'checkpoint_last.pt')]
    run([sys.executable, '-m', 'core.tools.nexar_eval', *runs, '--census', CENSUS, '--clip-s1-dir', L_VAL_X,
         '--stats-dir', L_E3_STATS, '--video-s1-dir', L_VAL_U, '--data-dir', L_DATA['whole'],
         '--split', constants.V2_SPLIT_NEXAR_VAL, '--out-dir', ZS_OUT], log=LOCAL / 'zeroshot_val.log')
zs = json.loads((ZS_OUT / 'nexar_readout.json').read_text())
print('zero-shot E3 A3 crop endpoint:', zs['crops']['mean_over_placements']['macro'])

## 6. Second look on DoTA-CAP-dev (opened set — descriptive only, D-N8)

Protocol B exactly as E3 read A0/A3, with the **Nexar** statistics baked on DoTA-CAP for A3. This set was opened by
E3: the number says how a Nexar-trained model transfers, it is not a clean endpoint and decides nothing.

In [ ]:
from core.tools import protocol_b_eval

cap_dev = load_split(constants.V2_SPLIT_DOTA_CAP_DEV)
L_DOTA_S1 = stage(DOTA_S1, LOCAL / 'stage' / 'DoTA_s1_cap_dev', cap_dev)
L_DOTA_LABELS = stage(DOTA / 'labels_s8' / constants.DEFS_FILENAME,
                      LOCAL / 'stage' / 'dota_labels' / constants.DEFS_FILENAME).parent
if not (A3_CAP / constants.V2_INPUT_MANIFEST_FILENAME).exists():
    cap_ids = write_ids(LOCAL / 'ids' / 'dota_cap_dev.txt', cap_dev)
    run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'apply', '--stats-dir', A3_BAKE,
         '--clip-dir', L_DOTA_S1, '--video-dir', CAP_S1, '--stride', constants.NEXAR_EVAL_STRIDE,
         '--ids-file', cap_ids, '--out-dir', A3_CAP])
L_A3_CAP = stage(A3_CAP, LOCAL / 'stage' / 'A3_cap_nexar')
CAP_IN = {'A0': L_DOTA_S1, 'A3': L_A3_CAP}
CAP_OUT = {(c, a): REPORTS / 'nexar_dota_cap' / f'{TAG}_{c}_{a}' for c in CONSTRUCTIONS for a in ARMS}
for (c, arm), out in CAP_OUT.items():
    if not all(verified(c, run_dir(c, arm, s)) for s in SEEDS):
        continue
    if (out / protocol_b_eval.CLIP_SCORES_NPZ).exists():
        print(f'{c}/{arm}: reusing')
        continue
    runs = [x for s in SEEDS for x in ('--run', f's{s}', run_dir(c, arm, s) / 'checkpoint_last.pt')]
    run([sys.executable, '-m', 'core.tools.protocol_b_eval', *runs, '--input-dir', CAP_IN[arm],
         '--s1-dir', L_DOTA_S1, '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
         '--data-dir', L_DOTA_LABELS, '--split', constants.V2_SPLIT_DOTA_CAP_DEV, '--out-dir', out],
        log=LOCAL / f'{c}_{arm}_cap.log')

## 7. Summary + window trigger → paste `summary_<tag>.md` to Claude

The trigger (D-N10, W1–W3) is computed on the **pilot** only and decides one thing: whether `window` is trained.

In [ ]:
def fmt(ci):
    return '—' if not ci else f"{ci['mean']:.4f} [{ci['low']:.4f}, {ci['high']:.4f}]"


lines = [f'# Nexar {TAG} summary (seeds {SEEDS})', '',
         f"val middle ruler (whole videos, position alone): {build['val_middle_ruler_macro']}", '',
         '| construction | arm | crops mean (endpoint) | crop @0.1 | @0.5 | @0.9 | whole macro | whole micro | clip AUC | DoTA-CAP-dev (2nd look) |',
         '|---|---|---|---|---|---|---|---|---|---|']
for c in CONSTRUCTIONS:
    for arm in ARMS:
        vj, cj = VAL_OUT[(c, arm)] / 'nexar_readout.json', CAP_OUT[(c, arm)] / protocol_b_eval.READOUT_JSON
        if not vj.exists():
            continue
        r = json.loads(vj.read_text())
        cap = json.loads(cj.read_text())['macro'] if cj.exists() else None
        cr, w = r['crops'], r['whole']
        lines.append(f"| {c} | {arm} | **{fmt(cr['mean_over_placements']['macro'])}** | {fmt(cr['0.1']['macro'])} | "
                     f"{fmt(cr['0.5']['macro'])} | {fmt(cr['0.9']['macro'])} | {fmt(w['macro'])} | {w['micro']:.4f} | "
                     f"{w['clip_auc_max']:.4f} | {fmt(cap)} |")
ruler = json.loads(next(iter(VAL_OUT.values())).joinpath('nexar_readout.json').read_text())['crops']
zsr = zs['crops']
lines.append(f"| zero-shot (E3 T2) | A3 | **{fmt(zsr['mean_over_placements']['macro'])}** | {fmt(zsr['0.1']['macro'])} | "
             f"{fmt(zsr['0.5']['macro'])} | {fmt(zsr['0.9']['macro'])} | {fmt(zs['whole']['macro'])} | "
             f"{zs['whole']['micro']:.4f} | {zs['whole']['clip_auc_max']:.4f} | (E3) |")
lines += ['', 'middle ruler per crop placement: ' +
          ', '.join(f"{p}: {ruler[str(p)]['middle_ruler_macro']['mean']:.3f}" for p in constants.NEXAR_CROP_PLACEMENTS)]
for c in CONSTRUCTIONS:
    for arm in ARMS:
        last = metrics_last(run_dir(c, arm, SEEDS[0]) / 'metrics.jsonl')
        if last:
            keep = {k: round(v, 4) for k, v in last.items() if isinstance(v, float)}
            lines.append(f'* last metrics {c}/{arm} s{SEEDS[0]}: step {last["global_step"]} {keep}')
if TAG == 'pilot' and all((VAL_OUT[('whole', a)] / 'nexar_readout.json').exists() for a in ARMS):
    TRIG = REPORTS / 'nexar_val' / 'trigger_pilot'
    run([sys.executable, '-m', 'core.tools.nexar_trigger', '--whole-a3', VAL_OUT[('whole', 'A3')],
         '--whole-a0', VAL_OUT[('whole', 'A0')], '--zero-shot', ZS_OUT, '--out-dir', TRIG])
    lines += ['', (TRIG / 'window_trigger.md').read_text()]
summary = '\n'.join(lines) + '\n'
(REPORTS / 'nexar_val').mkdir(parents=True, exist_ok=True)
(REPORTS / 'nexar_val' / f'summary_{TAG}.md').write_text(summary)
print(summary)